<a href="https://colab.research.google.com/github/ParthV303/Visual-Question-Answering-for-Documents/blob/main/DocVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries

!pip install -q torch torchvision torchaudio
!pip install -q transformers accelerate
!pip install -q qwen-vl-utils
!pip install -q pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 23.9 MB/s eta 0:00:00


In [2]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.5 MB/s eta 0:00:00


In [3]:
%%writefile app.py

import streamlit as st
import pandas as pd
import torch
import time
import gc

from PIL import Image

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor
)

from qwen_vl_utils import process_vision_info

# ----------------------------------
# Page Configuration
# ----------------------------------

st.set_page_config(
    page_title="DocVQA",
    page_icon="📄",
    layout="wide"
)

# ----------------------------------
# Load Model
# ----------------------------------

@st.cache_resource
def load_model():

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct",
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    processor = AutoProcessor.from_pretrained(
        "Qwen/Qwen2.5-VL-3B-Instruct"
    )

    return model, processor


model, processor = load_model()

# ----------------------------------
# Title
# ----------------------------------

st.title("DocVQA System")

st.success("Qwen2.5-VL loaded successfully!")

st.write("Upload one or more document images and ask a question.")

st.divider()

# ----------------------------------
# Upload Images
# ----------------------------------

uploaded_files = st.file_uploader(
    "Upload Document Images",
    type=["png", "jpg", "jpeg"],
    accept_multiple_files=True
)

# ----------------------------------
# Display Images
# ----------------------------------

if uploaded_files:

    st.success(f"{len(uploaded_files)} image(s) uploaded successfully!")

    cols = st.columns(min(3, len(uploaded_files)))

    for i, file in enumerate(uploaded_files):

        with cols[i % len(cols)]:

            st.image(
                file,
                caption=file.name,
                 width=200
            )

st.divider()

# ----------------------------------
# Question
# ----------------------------------

question = st.text_input(
    "Enter your question",
    placeholder="Example: What is the invoice date?"
)

# ----------------------------------
# Get Answers
# ----------------------------------

if st.button("🚀 Get Answers"):

    if not uploaded_files:

        st.warning("Please upload at least one image.")

    elif question.strip() == "":

        st.warning("Please enter a question.")

    else:

        results = []

        with st.spinner("Generating answers..."):

            for uploaded_file in uploaded_files:

                image = Image.open(uploaded_file).convert("RGB")
                image.thumbnail((896 , 896))

                start_time = time.time()

                try:

                    messages = [
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "image",
                                    "image": image,
                                },
                                {
                                    "type": "text",
                                    "text": question,
                                },
                            ],
                        }
                    ]

                    # Prompt
                    text = processor.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True
                    )

                    # Vision Inputs
                    image_inputs, video_inputs = process_vision_info(messages)

                    # Processor
                    inputs = processor(
                        text=[text],
                        images=image_inputs,
                        videos=video_inputs,
                        padding=True,
                        return_tensors="pt",
                    )

                    inputs = inputs.to(model.device)

                    # Inference
                    with torch.inference_mode():

                        generated_ids = model.generate(
                            **inputs,
                            max_new_tokens=128
                        )

                    # Remove prompt tokens
                    generated_ids_trimmed = [
                        output_ids[len(input_ids):]
                        for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
                    ]

                    # Decode answer
                    answer = processor.batch_decode(
                        generated_ids_trimmed,
                        skip_special_tokens=True,
                        clean_up_tokenization_spaces=False
                    )[0].strip()

                    end_time = time.time()

                    results.append({

                        "file": uploaded_file.name,
                        "answer": answer,
                        "time": round(end_time - start_time, 2)

                    })

                except Exception as e:

                    results.append({

                        "file": uploaded_file.name,
                        "answer": str(e),
                        "time": 0

                    })

                finally:

                    if "image" in locals():
                        del image

                    if "image_inputs" in locals():
                        del image_inputs

                    if "video_inputs" in locals():
                        del video_inputs

                    if "inputs" in locals():
                        del inputs

                    if "generated_ids" in locals():
                        del generated_ids

                    if "generated_ids_trimmed" in locals():
                        del generated_ids_trimmed

                    torch.cuda.empty_cache()
                    gc.collect()

        st.success("Batch Processing Completed!")

        st.divider()

        st.header("Results")

        for result in results:

            st.subheader(result["file"])

            st.write("**Answer:**")

            st.success(result["answer"])

            st.caption(f"Time : {result['time']} sec")
            import pandas as pd

        df = pd.DataFrame(results)

        csv = df.to_csv(index=False).encode("utf-8")

        st.download_button(
            label="Download Results CSV",
            data=csv,
            file_name="batch_docvqa_results.csv",
            mime="text/csv"
        )



Writing app.py


In [4]:
%%writefile model_loader.py

model = ...

processor = ...

Writing model_loader.py


In [5]:
!cat model_loader.py


model = ...

processor = ...


In [6]:
!pkill -f streamlit

In [7]:
!streamlit run app.py &>/content/logs.txt &

In [8]:
!pip install pyngrok

In [9]:
from pyngrok import ngrok

ngrok.set_auth_token("3EexzzFHV45cmror8toSW6nddVx_6SdH3Hqp1Ybfquhagin2v")

In [10]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://only-treat-safehouse.ngrok-free.dev" -> "http://localhost:8501"
